# Corndel AI6 (Level 6 ML/AI Engineer)


---


# Unit **7** Module **4**: **From Metrics to Decisions**
## NOTEBOOK: _The Fault That Wasn't Found_

---

<br>
Welcome to the notebook that supports your learning in this module:

| | |
|:--|:--|
| **Context** | Rail axle bearing inspection.<br>Bearings develop faults gradually, for example through fatigue or cracking.<br>Caught early: a scheduled replacement.<br>Missed: axle failure with possible severe liability. |
| **Why this domain** | It's full of trade-offs so metrics support decisions.<br>A defect has a clear cost... but so does a false alarm.<br>This domain exposes you to how an AI Engineer must navigate complex contexts. |
| **The system** | A classifier is needed, as it flags images for human review.<br>A domain-specialist engineer makes the final call (relax - not you!). |
| **Your task** | You recommend which of two candidate models to deploy and why. |

You have two candidate computer vision models. Both were trained to classify bearing inspection images as **defect** or **no defect**. Both were evaluated on the same holdout set. One will be recommended for deployment. That recommendation is yours to make (and justify!). This notebook walks you through the evidence and asks you to do something that no confusion matrix can do for you.

> **How to work through this notebook**  
> Run each code cell in order using Shift+Enter. In Sections 1 and 2 you will run and interpret code. In Section 3 you will write before the notebook proceeds. In Section 4 you will write your deployment recommendation. Section 5 is optional.

*Estimated time: 45 minutes.*

---

## Setup

*Run this cell first. It loads the libraries and helper functions used throughout.*

In [ ]:
import io
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

# ── Helper: confusion matrix metrics ────────────────────────────────────
def confusion_metrics(y_true, y_prob, threshold=0.5):
    """Return TP, FP, FN, TN and derived metrics at a given threshold."""
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    tp = int(((y_pred==1) & (y_true==1)).sum())
    fp = int(((y_pred==1) & (y_true==0)).sum())
    fn = int(((y_pred==0) & (y_true==1)).sum())
    tn = int(((y_pred==0) & (y_true==0)).sum())
    n  = len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    return dict(TP=tp, FP=fp, FN=fn, TN=tn,
                Accuracy=(tp+tn)/n, Precision=prec, Recall=rec, F1=f1)

# ── Helper: precision-recall curve (no sklearn dependency) ───────────────
def pr_curve(y_true, y_prob, steps=300):
    """Return arrays of (precision, recall, threshold) across threshold range."""
    thresholds = np.linspace(0.01, 0.99, steps)
    precs, recs = [], []
    for t in thresholds:
        m = confusion_metrics(y_true, y_prob, t)
        precs.append(m['Precision'])
        recs.append(m['Recall'])
    return np.array(precs), np.array(recs), thresholds

# ── Helper: word-count gate ───────────────────────────────────────────────
def check_response(label, text, min_words):
    """Raise an error if the response is a placeholder or too short."""
    placeholder_phrases = [
        'replace this', 'your answer', 'write here', 'answer here',
        'type here', 'enter your', 'insert your'
    ]
    lower = text.lower().strip()
    if not lower:
        raise ValueError(f'\n[{label}] Response is empty. Write your answer before running this cell.')
    for phrase in placeholder_phrases:
        if phrase in lower:
            raise ValueError(f'\n[{label}] The placeholder text is still there. Replace it with your own writing.')
    word_count = len(text.split())
    if word_count < min_words:
        raise ValueError(
            f'\n[{label}] Your response is {word_count} word(s). '
            f'This question needs at least {min_words} words. '
            f'Be more specific.'
        )
    return text.strip()

print('Setup complete.')

---

## Section 1: Load and Explore

*About 10 minutes.*

The data below is the pre-computed evaluation set: 500 bearing inspection images, of which 100 contain confirmed defects. The columns record the true label and each model's predicted probability that the image contains a defect.

Run the cells in order.

In [ ]:
# Pre-computed evaluation results — 500-image holdout set
# true_label  : 1 = defect confirmed, 0 = no defect
# model_a_prob : Model A predicted probability of defect
# model_b_prob : Model B predicted probability of defect

CSV_DATA = \
"""image_id,true_label,model_a_prob,model_b_prob
img_0000,0,0.1938,0.2377
img_0001,1,0.1933,0.1400
img_0002,0,0.2124,0.0265
img_0003,0,0.1798,0.0447
img_0004,1,0.2008,0.9605
img_0005,0,0.0604,0.0866
img_0006,0,0.0823,0.0662
img_0007,0,0.0952,0.1593
img_0008,0,0.0517,0.6364
img_0009,0,0.0668,0.1506
img_0010,0,0.0674,0.0693
img_0011,0,0.0716,0.0501
img_0012,0,0.0737,0.0302
img_0013,0,0.1162,0.1317
img_0014,0,0.1729,0.0951
img_0015,0,0.1259,0.0739
img_0016,0,0.1400,0.1679
img_0017,0,0.1967,0.0919
img_0018,0,0.1004,0.1613
img_0019,0,0.0633,0.2058
img_0020,0,0.2139,0.0550
img_0021,0,0.1206,0.0502
img_0022,0,0.0738,0.2507
img_0023,0,0.1173,0.1347
img_0024,0,0.1320,0.2458
img_0025,1,0.8940,0.8693
img_0026,1,0.3056,0.8636
img_0027,0,0.1710,0.1364
img_0028,0,0.0294,0.0617
img_0029,0,0.1640,0.2356
img_0030,0,0.0367,0.0671
img_0031,1,0.8535,0.9743
img_0032,1,0.1960,0.8735
img_0033,0,0.0839,0.0806
img_0034,0,0.1169,0.0873
img_0035,0,0.0217,0.0677
img_0036,0,0.0611,0.0717
img_0037,0,0.2366,0.1290
img_0038,0,0.0753,0.1216
img_0039,1,0.4068,0.8395
img_0040,1,0.2155,0.9447
img_0041,1,0.0667,0.9482
img_0042,0,0.0598,0.0706
img_0043,0,0.0591,0.1155
img_0044,0,0.0531,0.0790
img_0045,0,0.0254,0.0495
img_0046,0,0.2399,0.0270
img_0047,0,0.1003,0.0552
img_0048,0,0.0353,0.0921
img_0049,0,0.3493,0.1653
img_0050,0,0.0990,0.0514
img_0051,0,0.1110,0.0933
img_0052,0,0.0485,0.0541
img_0053,0,0.1554,0.1478
img_0054,0,0.2153,0.0909
img_0055,0,0.0392,0.1410
img_0056,0,0.1487,0.2047
img_0057,0,0.1026,0.1142
img_0058,0,0.1558,0.1434
img_0059,0,0.0976,0.2065
img_0060,0,0.0868,0.0260
img_0061,0,0.1900,0.2915
img_0062,0,0.0989,0.1097
img_0063,0,0.0157,0.0832
img_0064,1,0.9674,0.9486
img_0065,0,0.3591,0.0616
img_0066,1,0.2922,0.9708
img_0067,0,0.1418,0.0765
img_0068,0,0.0814,0.7081
img_0069,0,0.2273,0.0706
img_0070,0,0.0537,0.2439
img_0071,0,0.1013,0.1846
img_0072,0,0.0851,0.7503
img_0073,1,0.9318,0.8954
img_0074,0,0.1719,0.0713
img_0075,1,0.8934,0.8546
img_0076,0,0.1674,0.0684
img_0077,0,0.0372,0.1291
img_0078,0,0.1446,0.1011
img_0079,0,0.0430,0.1325
img_0080,0,0.1452,0.1289
img_0081,0,0.1255,0.7359
img_0082,0,0.0984,0.0842
img_0083,0,0.0259,0.1424
img_0084,0,0.1271,0.2027
img_0085,0,0.1426,0.3356
img_0086,0,0.0593,0.0479
img_0087,0,0.1189,0.0637
img_0088,1,0.9621,0.9360
img_0089,0,0.1093,0.1826
img_0090,0,0.1175,0.1442
img_0091,0,0.1252,0.0910
img_0092,0,0.0857,0.5844
img_0093,0,0.1816,0.1619
img_0094,0,0.1349,0.1966
img_0095,0,0.1055,0.1318
img_0096,1,0.9469,0.9279
img_0097,0,0.0638,0.1463
img_0098,1,0.9015,0.9900
img_0099,1,0.7148,0.9870
img_0100,0,0.0893,0.1645
img_0101,0,0.1505,0.2874
img_0102,0,0.1427,0.1646
img_0103,0,0.0322,0.2664
img_0104,0,0.0409,0.0526
img_0105,0,0.0509,0.1171
img_0106,0,0.1381,0.2347
img_0107,0,0.1891,0.1698
img_0108,0,0.0881,0.1588
img_0109,1,0.0988,0.1441
img_0110,0,0.0159,0.1529
img_0111,1,0.8477,0.9819
img_0112,0,0.2027,0.6720
img_0113,0,0.1085,0.2070
img_0114,0,0.1135,0.0725
img_0115,0,0.1776,0.1527
img_0116,0,0.0502,0.0773
img_0117,0,0.0823,0.1003
img_0118,0,0.1164,0.1294
img_0119,1,0.9418,0.9860
img_0120,0,0.1358,0.2023
img_0121,1,0.8738,0.9444
img_0122,1,0.1972,0.1061
img_0123,0,0.0415,0.1393
img_0124,0,0.0491,0.0572
img_0125,0,0.0853,0.0547
img_0126,1,0.3227,0.9086
img_0127,0,0.1527,0.0719
img_0128,1,0.8038,0.9525
img_0129,0,0.1268,0.0305
img_0130,0,0.1150,0.1934
img_0131,0,0.0424,0.2101
img_0132,0,0.1360,0.1101
img_0133,0,0.0344,0.0569
img_0134,0,0.2075,0.1002
img_0135,0,0.1573,0.0997
img_0136,0,0.2462,0.1938
img_0137,0,0.0355,0.1162
img_0138,0,0.1465,0.1058
img_0139,0,0.1076,0.0946
img_0140,0,0.1218,0.1659
img_0141,0,0.1088,0.1292
img_0142,0,0.2171,0.1965
img_0143,0,0.0657,0.0659
img_0144,0,0.1138,0.2189
img_0145,1,0.9011,0.9037
img_0146,1,0.7565,0.9879
img_0147,1,0.8124,0.9310
img_0148,0,0.0455,0.2441
img_0149,0,0.0759,0.0348
img_0150,0,0.1433,0.0864
img_0151,0,0.1435,0.1093
img_0152,0,0.0231,0.3370
img_0153,1,0.1234,0.1036
img_0154,0,0.1875,0.0557
img_0155,0,0.1353,0.1813
img_0156,0,0.6823,0.5873
img_0157,0,0.1261,0.1703
img_0158,1,0.9889,0.9064
img_0159,0,0.0993,0.1200
img_0160,0,0.0814,0.1661
img_0161,0,0.0903,0.0781
img_0162,0,0.1112,0.1849
img_0163,0,0.2658,0.0941
img_0164,0,0.1829,0.6974
img_0165,0,0.1945,0.1268
img_0166,0,0.1072,0.0894
img_0167,0,0.0826,0.5830
img_0168,1,0.8255,0.8649
img_0169,1,0.8881,0.9115
img_0170,0,0.1369,0.0770
img_0171,0,0.1534,0.0473
img_0172,0,0.2272,0.0372
img_0173,0,0.0872,0.1433
img_0174,1,0.9421,0.9723
img_0175,0,0.0999,0.0693
img_0176,0,0.0875,0.0741
img_0177,1,0.9138,0.8956
img_0178,0,0.1186,0.1765
img_0179,0,0.2180,0.0447
img_0180,0,0.1044,0.0248
img_0181,0,0.0417,0.0348
img_0182,0,0.2501,0.0781
img_0183,0,0.1882,0.1048
img_0184,0,0.0256,0.1099
img_0185,0,0.2440,0.5852
img_0186,0,0.1120,0.1117
img_0187,0,0.1274,0.2498
img_0188,0,0.0805,0.1700
img_0189,1,0.9818,0.9394
img_0190,0,0.1641,0.0806
img_0191,0,0.0697,0.2390
img_0192,0,0.0482,0.0294
img_0193,0,0.1178,0.7322
img_0194,0,0.1039,0.1619
img_0195,0,0.1019,0.2060
img_0196,0,0.1900,0.2766
img_0197,1,0.9319,0.9813
img_0198,0,0.0755,0.2020
img_0199,0,0.1349,0.0878
img_0200,0,0.0589,0.1519
img_0201,0,0.1716,0.0889
img_0202,0,0.0582,0.7089
img_0203,0,0.0599,0.1592
img_0204,0,0.1419,0.1290
img_0205,0,0.2090,0.0498
img_0206,0,0.2243,0.1285
img_0207,0,0.0702,0.2295
img_0208,0,0.1594,0.0882
img_0209,0,0.1190,0.0479
img_0210,0,0.2054,0.0747
img_0211,0,0.1857,0.2317
img_0212,1,0.2648,0.8494
img_0213,0,0.0547,0.1022
img_0214,1,0.8616,0.8536
img_0215,0,0.2275,0.1292
img_0216,0,0.0226,0.0692
img_0217,1,0.1416,0.9239
img_0218,1,0.1357,0.1454
img_0219,0,0.1661,0.0778
img_0220,0,0.1227,0.1682
img_0221,0,0.1297,0.1422
img_0222,0,0.0217,0.2538
img_0223,0,0.0436,0.1071
img_0224,0,0.1244,0.1403
img_0225,0,0.2225,0.1126
img_0226,0,0.1575,0.2152
img_0227,0,0.1903,0.0796
img_0228,1,0.8782,0.7363
img_0229,1,0.8562,0.9401
img_0230,0,0.1022,0.1141
img_0231,0,0.1608,0.1841
img_0232,0,0.1440,0.0389
img_0233,0,0.2022,0.0844
img_0234,1,0.2898,0.8089
img_0235,0,0.0171,0.0618
img_0236,0,0.1090,0.2416
img_0237,0,0.1490,0.1884
img_0238,0,0.0795,0.0730
img_0239,1,0.8859,0.9570
img_0240,0,0.0857,0.1875
img_0241,0,0.1758,0.2124
img_0242,1,0.7836,0.8643
img_0243,0,0.0694,0.1720
img_0244,0,0.1674,0.3504
img_0245,0,0.1426,0.3319
img_0246,0,0.0767,0.1357
img_0247,0,0.0981,0.0617
img_0248,1,0.1867,0.1051
img_0249,1,0.3219,0.1147
img_0250,1,0.8519,0.8789
img_0251,0,0.1309,0.0702
img_0252,0,0.2242,0.0586
img_0253,1,0.7836,0.8575
img_0254,0,0.1141,0.0852
img_0255,0,0.0882,0.2156
img_0256,0,0.0653,0.2502
img_0257,0,0.1171,0.0547
img_0258,1,0.9788,0.9441
img_0259,0,0.0542,0.1029
img_0260,0,0.0143,0.3227
img_0261,1,0.7850,0.9573
img_0262,0,0.0874,0.1518
img_0263,0,0.0533,0.1203
img_0264,0,0.1135,0.0835
img_0265,0,0.0155,0.0936
img_0266,0,0.0952,0.1171
img_0267,1,0.9433,0.8421
img_0268,0,0.0326,0.0979
img_0269,0,0.0789,0.1579
img_0270,0,0.0909,0.1576
img_0271,1,0.7879,0.9393
img_0272,0,0.0708,0.1822
img_0273,0,0.3013,0.0975
img_0274,0,0.1346,0.1125
img_0275,0,0.1516,0.1934
img_0276,1,0.1524,0.9630
img_0277,0,0.1576,0.1726
img_0278,0,0.0833,0.1200
img_0279,0,0.1616,0.1679
img_0280,0,0.0868,0.0530
img_0281,0,0.0409,0.2598
img_0282,0,0.0816,0.1848
img_0283,0,0.2039,0.1039
img_0284,0,0.1879,0.0783
img_0285,1,0.9218,0.8895
img_0286,0,0.0183,0.1348
img_0287,0,0.0617,0.0627
img_0288,0,0.0325,0.1915
img_0289,0,0.0513,0.0872
img_0290,0,0.0517,0.1369
img_0291,0,0.1015,0.1675
img_0292,0,0.0247,0.0989
img_0293,0,0.1660,0.6431
img_0294,1,0.8406,0.8975
img_0295,0,0.0407,0.1774
img_0296,0,0.0947,0.1234
img_0297,1,0.8711,0.9274
img_0298,0,0.0905,0.1751
img_0299,0,0.1035,0.1303
img_0300,0,0.0770,0.0434
img_0301,0,0.1031,0.0324
img_0302,0,0.1654,0.1015
img_0303,0,0.0249,0.0849
img_0304,0,0.0920,0.1120
img_0305,0,0.1866,0.0991
img_0306,0,0.2747,0.0337
img_0307,0,0.0531,0.1255
img_0308,1,0.8128,0.8861
img_0309,0,0.0142,0.1502
img_0310,0,0.0718,0.1639
img_0311,1,0.9135,0.8620
img_0312,0,0.0410,0.0638
img_0313,0,0.0388,0.2530
img_0314,0,0.0642,0.1893
img_0315,0,0.0963,0.2425
img_0316,0,0.2159,0.1340
img_0317,1,0.1861,0.0710
img_0318,1,0.9441,0.9408
img_0319,0,0.0997,0.0587
img_0320,0,0.0627,0.0766
img_0321,0,0.1066,0.1957
img_0322,0,0.1850,0.0586
img_0323,0,0.1675,0.0471
img_0324,1,0.8396,0.7844
img_0325,0,0.0383,0.1153
img_0326,0,0.0428,0.0635
img_0327,0,0.1490,0.1444
img_0328,0,0.0966,0.0379
img_0329,0,0.0413,0.1935
img_0330,0,0.0325,0.1044
img_0331,1,0.2020,0.8554
img_0332,0,0.1137,0.0661
img_0333,0,0.0763,0.1245
img_0334,1,0.9315,0.8936
img_0335,0,0.0963,0.0550
img_0336,0,0.0838,0.0939
img_0337,0,0.2029,0.0963
img_0338,1,0.9499,0.9682
img_0339,1,0.0844,0.8970
img_0340,0,0.1087,0.1905
img_0341,0,0.0873,0.1115
img_0342,1,0.8758,0.8490
img_0343,0,0.0265,0.1373
img_0344,0,0.0715,0.1710
img_0345,0,0.0840,0.1451
img_0346,0,0.1142,0.0604
img_0347,0,0.1034,0.0793
img_0348,0,0.0841,0.1881
img_0349,0,0.1073,0.0964
img_0350,0,0.0213,0.1552
img_0351,0,0.0925,0.0589
img_0352,0,0.1688,0.1542
img_0353,0,0.0354,0.0831
img_0354,0,0.1860,0.1077
img_0355,0,0.0917,0.0790
img_0356,0,0.0613,0.1590
img_0357,0,0.0630,0.1976
img_0358,0,0.1440,0.1644
img_0359,0,0.0165,0.1373
img_0360,1,0.1312,0.2011
img_0361,0,0.1143,0.0674
img_0362,0,0.1388,0.6654
img_0363,0,0.1724,0.1541
img_0364,0,0.0938,0.0958
img_0365,0,0.1244,0.0246
img_0366,0,0.0957,0.1057
img_0367,0,0.2233,0.1136
img_0368,0,0.2022,0.6949
img_0369,0,0.0986,0.1556
img_0370,0,0.0563,0.0842
img_0371,1,0.8386,0.7435
img_0372,0,0.0414,0.1539
img_0373,0,0.0901,0.1016
img_0374,0,0.0928,0.0818
img_0375,1,0.9102,0.9195
img_0376,0,0.0625,0.0881
img_0377,0,0.0534,0.1251
img_0378,0,0.0671,0.1339
img_0379,0,0.1873,0.0812
img_0380,0,0.1399,0.7700
img_0381,0,0.1007,0.1982
img_0382,0,0.2054,0.0977
img_0383,0,0.0402,0.0582
img_0384,0,0.0496,0.1166
img_0385,0,0.1338,0.1621
img_0386,0,0.1105,0.1203
img_0387,0,0.0958,0.7970
img_0388,0,0.1816,0.1387
img_0389,1,0.2357,0.9753
img_0390,0,0.0692,0.1586
img_0391,1,0.8427,0.9197
img_0392,1,0.0663,0.9463
img_0393,0,0.0644,0.1869
img_0394,0,0.1110,0.1747
img_0395,0,0.2602,0.2098
img_0396,1,0.1444,0.9177
img_0397,0,0.1339,0.2369
img_0398,0,0.1469,0.0343
img_0399,1,0.9760,0.8891
img_0400,0,0.1134,0.0981
img_0401,0,0.2168,0.2579
img_0402,0,0.0538,0.1251
img_0403,1,0.7769,0.8249
img_0404,0,0.0565,0.5434
img_0405,0,0.1942,0.0609
img_0406,0,0.0459,0.1508
img_0407,0,0.1501,0.1155
img_0408,0,0.0405,0.0646
img_0409,1,0.2238,0.9802
img_0410,0,0.0428,0.1348
img_0411,1,0.2093,0.9535
img_0412,0,0.1520,0.1302
img_0413,0,0.0845,0.0860
img_0414,1,0.8986,0.9019
img_0415,0,0.0760,0.1344
img_0416,1,0.1515,0.9271
img_0417,0,0.1357,0.2084
img_0418,1,0.9665,0.8500
img_0419,0,0.1195,0.2064
img_0420,1,0.2836,0.8861
img_0421,0,0.1796,0.2502
img_0422,1,0.8882,0.8848
img_0423,0,0.1314,0.1427
img_0424,1,0.9697,0.9414
img_0425,0,0.1913,0.2515
img_0426,1,0.9594,0.9900
img_0427,0,0.2092,0.2099
img_0428,1,0.2337,0.1541
img_0429,1,0.9610,0.7699
img_0430,0,0.2215,0.0351
img_0431,0,0.1847,0.0289
img_0432,1,0.7777,0.8225
img_0433,0,0.0899,0.1861
img_0434,0,0.2596,0.1340
img_0435,0,0.1953,0.1427
img_0436,0,0.1613,0.1216
img_0437,0,0.1807,0.2297
img_0438,0,0.1726,0.1612
img_0439,0,0.0610,0.2919
img_0440,0,0.0691,0.2333
img_0441,0,0.0325,0.0423
img_0442,1,0.9007,0.8746
img_0443,0,0.1525,0.2011
img_0444,0,0.1621,0.1692
img_0445,0,0.1548,0.0400
img_0446,0,0.1703,0.1176
img_0447,1,0.2276,0.9610
img_0448,0,0.2876,0.1822
img_0449,0,0.0513,0.0997
img_0450,0,0.0465,0.1869
img_0451,0,0.0538,0.1279
img_0452,0,0.1874,0.0368
img_0453,0,0.1117,0.0992
img_0454,1,0.8751,0.9900
img_0455,0,0.2110,0.0469
img_0456,0,0.0501,0.1759
img_0457,1,0.9438,0.8796
img_0458,1,0.9627,0.9683
img_0459,0,0.2456,0.2230
img_0460,0,0.0310,0.1908
img_0461,0,0.1243,0.0353
img_0462,0,0.1132,0.0462
img_0463,0,0.0237,0.0912
img_0464,0,0.6699,0.6686
img_0465,0,0.1262,0.0567
img_0466,0,0.2053,0.1652
img_0467,0,0.0417,0.1704
img_0468,0,0.0778,0.1488
img_0469,0,0.2231,0.0761
img_0470,0,0.0149,0.2191
img_0471,1,0.2466,0.9267
img_0472,1,0.1953,0.8355
img_0473,1,0.7952,0.7969
img_0474,0,0.3091,0.1101
img_0475,0,0.0707,0.1933
img_0476,1,0.7341,0.9135
img_0477,0,0.1681,0.2129
img_0478,0,0.2274,0.1265
img_0479,0,0.1081,0.1124
img_0480,0,0.0877,0.0761
img_0481,1,0.9112,0.9324
img_0482,0,0.1679,0.2194
img_0483,0,0.0353,0.0803
img_0484,1,0.9083,0.9823
img_0485,0,0.6031,0.7698
img_0486,1,0.7727,0.9544
img_0487,0,0.1927,0.1347
img_0488,0,0.0442,0.0238
img_0489,0,0.1279,0.0731
img_0490,0,0.1434,0.0578
img_0491,0,0.2018,0.1008
img_0492,1,0.8813,0.9176
img_0493,0,0.1391,0.7207
img_0494,0,0.0854,0.1124
img_0495,0,0.1780,0.1056
img_0496,0,0.0901,0.2959
img_0497,0,0.0758,0.2089
img_0498,0,0.3155,0.1821
img_0499,0,0.5586,0.5600"""

df = pd.read_csv(io.StringIO(CSV_DATA))

print(f'Holdout set: {len(df)} images')
print(f'  With defects     : {df.true_label.sum()}')
print(f'  Without defects  : {(df.true_label==0).sum()}')
df.head(6)

### 1.1 Compute the metrics

Run the cell below to compute accuracy, precision, recall and F1 for both models at the default decision threshold of 0.5.

A predicted probability at or above 0.5 is classified as a defect.

In [ ]:
y  = df.true_label.values
pa = df.model_a_prob.values
pb = df.model_b_prob.values

ma = confusion_metrics(y, pa)
mb = confusion_metrics(y, pb)

summary = pd.DataFrame({'Model A': ma, 'Model B': mb})

print('Confusion matrix counts')
display(summary.loc[['TP','FP','FN','TN']].astype(int))
print()
print('Derived metrics')
display(summary.loc[['Accuracy','Precision','Recall','F1']].map(lambda x: f'{x:.3f}'))

### 1.2 Visualise the confusion matrices

The cell below produces a visual layout of both matrices. Look at the false negative cell for each model. That is the number that will matter most as you work through this notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('Confusion matrices at threshold 0.5  —  500-image holdout set',
             fontsize=12, fontweight='bold', y=1.02)

labels_grid = [['True Negative\n(correctly cleared)', 'False Positive\n(unnecessary flag)'],
               ['False Negative\n(missed defect)', 'True Positive\n(defect caught)']]

cell_colours = [['#f0f7f0', '#e8f4fd'], ['#fdecea', '#e8f8f0']]

for ax, m, name in zip(axes, [ma, mb], ['Model A', 'Model B']):
    grid_vals = [[m['TN'], m['FP']], [m['FN'], m['TP']]]
    for i in range(2):
        for j in range(2):
            rect = plt.Rectangle([j-0.5, i-0.5], 1, 1, linewidth=2,
                                  edgecolor='#cccccc', facecolor=cell_colours[i][j])
            ax.add_patch(rect)
            # Red border on FN cell
            if i==1 and j==0:
                rect_fn = plt.Rectangle([j-0.5, i-0.5], 1, 1, linewidth=3,
                                        edgecolor='#c0392b', facecolor='none', zorder=5)
                ax.add_patch(rect_fn)
            ax.text(j, i, str(grid_vals[i][j]), ha='center', va='center',
                    fontsize=20, fontweight='bold',
                    color='#c0392b' if (i==1 and j==0) else
                          '#27ae60' if (i==1 and j==1) else '#333')
            ax.text(j, i+0.32, labels_grid[i][j], ha='center', va='center',
                    fontsize=7.5, color='#666')

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Predicted\nno defect', 'Predicted\ndefect'], fontsize=9)
    ax.set_yticklabels(['Actual\nno defect', 'Actual\ndefect'], fontsize=9)
    ax.set_title(f'{name}\n'
                 f'Acc {m["Accuracy"]:.1%}  |  Prec {m["Precision"]:.1%}  |  Rec {m["Recall"]:.1%}',
                 fontsize=10, pad=8)
    ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.show()

print(f'Model A misses {ma["FN"]} defects. Model B misses {mb["FN"]} defects.')
print(f'Model A raises {ma["FP"]} false alarms. Model B raises {mb["FP"]} false alarms.')

Before moving on, make sure you can answer these questions for yourself:

- Why does Model A have higher accuracy than Model B on some metrics but not others?
- Which number in these matrices would you most want to reduce, and why?

You do not need to write anything yet. Take a moment, then run Section 2.

---

## Section 2: The Threshold Is Yours

*About 10 minutes.*

The confusion matrices above assume a decision threshold of 0.5. An image is flagged as a defect if the predicted probability is 0.5 or above. There is nothing fixed about 0.5. It is a default, not a constraint.

Changing the threshold changes the entire picture.

In [ ]:
# Precision-recall curves for both models across all possible thresholds

precs_a, recs_a, thresh_a = pr_curve(y, pa)
precs_b, recs_b, thresh_b = pr_curve(y, pb)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(recs_a, precs_a, color='#2a6ebb', linewidth=2, label='Model A', alpha=0.85)
ax.plot(recs_b, precs_b, color='#e67e22', linewidth=2.5, label='Model B')

# Default 0.5 operating points
ax.scatter([ma['Recall']], [ma['Precision']], color='#2a6ebb', s=90, zorder=6)
ax.scatter([mb['Recall']], [mb['Precision']], color='#e67e22', s=90, zorder=6)

ax.annotate(f'Model A at 0.5\n({ma["Precision"]:.0%} prec, {ma["Recall"]:.0%} rec)',
            xy=(ma['Recall'], ma['Precision']),
            xytext=(ma['Recall']-0.28, ma['Precision']-0.1),
            fontsize=8, color='#2a6ebb',
            arrowprops=dict(arrowstyle='->', color='#2a6ebb', lw=1))
ax.annotate(f'Model B at 0.5\n({mb["Precision"]:.0%} prec, {mb["Recall"]:.0%} rec)',
            xy=(mb['Recall'], mb['Precision']),
            xytext=(mb['Recall']-0.25, mb['Precision']-0.12),
            fontsize=8, color='#e67e22',
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1))

ax.set_xlabel('Recall  (fraction of actual defects caught)', fontsize=11)
ax.set_ylabel('Precision  (fraction of flags that are real defects)', fontsize=11)
ax.set_title('Precision vs recall at every possible threshold', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.08)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

Every point on these curves is a possible deployment configuration. Moving along the curve is not free: gaining precision costs recall, and vice versa.

| | |
|:--|:--|
| **Why does the curve have steps?** | The holdout set has 100 defects.<br>Each one caught moves recall by exactly 0.01.<br>A smooth curve needs thousands of positives. Yours does not have them. |
| **Why does Model B hold precision longer?** | Its probability scores are better calibrated for high-recall operation.<br>Model A starts trading precision for recall earlier.<br>That is why its default point sits at 66% recall despite high precision. |
| **What does the cliff tell you?** | The remaining defects look almost identical to clean bearings.<br>No threshold adjustment will catch them cleanly.<br>If those cases matter, the answer is a better model (not tuning the threshold!).
| **Is there a point where you would stop?** | Pushing recall from 90% to 100% costs a large number of false alarms.<br>Whether that is acceptable depends on who bears each type of error.<br>That is the question Section 3 asks you to answer. |

### 2.1 Threshold exploration task

Change `THRESHOLD_B` in the cell below and re-run it.

**Task:** Find a threshold for Model B at which it roughly matches Model A's default precision of 0.94. Then note what happens to recall when you get there.

In [ ]:
# Change this value and re-run
THRESHOLD_B = 0.50  # try values between 0.30 and 0.90

m_b_t = confusion_metrics(y, pb, threshold=THRESHOLD_B)

print(f'Model B at threshold {THRESHOLD_B:.2f}')
print(f'  Precision : {m_b_t["Precision"]:.3f}    (Model A at 0.5: {ma["Precision"]:.3f})')
print(f'  Recall    : {m_b_t["Recall"]:.3f}    (Model A at 0.5: {ma["Recall"]:.3f})')
print(f'  Accuracy  : {m_b_t["Accuracy"]:.3f}')
print()
print(f'  Missed defects  : {m_b_t["FN"]}  (Model A at 0.5: {ma["FN"]})')
print(f'  False alarms    : {m_b_t["FP"]}  (Model A at 0.5: {ma["FP"]})')

### 2.2 Putting the numbers in context

Run the cell below. It translates the false negative count into operational terms for a busy route.

In [ ]:
# Operational translation
# Assume the system processes 2,000 inspection images per week on one regional route
IMAGES_PER_WEEK = 2000
DEFECT_RATE     = df.true_label.mean()  # from our holdout set

expected_defects = IMAGES_PER_WEEK * DEFECT_RATE
fn_rate_a        = ma['FN'] / (ma['TP'] + ma['FN'])
fn_rate_b        = mb['FN'] / (mb['TP'] + mb['FN'])
missed_a         = expected_defects * fn_rate_a
missed_b         = expected_defects * fn_rate_b
extra_missed_a   = missed_a - missed_b

print(f'At {IMAGES_PER_WEEK:,} images per week (one route):')
print(f'  Expected defects per week         : {expected_defects:.0f}')
print()
print(f'  Model A: missed defects per week  : {missed_a:.1f}')
print(f'  Model B: missed defects per week  : {missed_b:.1f}')
print()
print(f'  Extra undetected defects per week with Model A  : {extra_missed_a:.1f}')
print(f'  Extra undetected defects per year with Model A  : {extra_missed_a*52:.0f}')

You now have the full technical picture.

You do not yet have a deployment recommendation. Section 3 explains why.

---

## Section 3: Before You Decide

*About 10 minutes.*

---

> **The notebook pauses here.**
>
> You have the confusion matrices. You have the threshold curves. You have the operational translation.
>
> You cannot make a deployment recommendation yet.
>
> Before this notebook accepts your decision, you must answer three questions. A response that stays inside metric language is not sufficient. This section asks you to look outward: away from the model, and towards the people affected by what it does.
>
> Replace the placeholder text in each variable below with your own writing. The cell will not run if you leave the placeholders in place or if your answer is too brief.

---

### Question 1

A false negative in this system means the model predicted *no defect* for a bearing that actually has one. That bearing enters service.

**Who is most at risk from a false negative? Name a specific person in a specific situation, not a general category.** _(Examples of 'wrong' answers: "Model A users", "A passenger on a busy train". Example 'correct' answer: "A commuter on the 07:22 from Gatwick aboard a Class 387 unit whose bearing has developed early-stage pitting that the model returned no defect for.")_

In [ ]:
person_at_risk = """Replace this text with your answer"""

# Minimum 15 words. Be specific: name a person in a situation, not a category.
person_at_risk = check_response('Person at risk', person_at_risk, min_words=15)

### Question 2

**Describe what happens to that person when the system produces a false negative.**

*Think through the sequence from missed defect to real-world consequence. Describe their experience concretely: what do they encounter, what do they not know, what happens to them?*

In [ ]:
consequence_of_false_negative = """Replace this text with your answer."""

# Minimum 25 words. Describe a sequence of events, not a category of harm.
consequence_of_false_negative = check_response('Consequence', consequence_of_false_negative, min_words=25)

### Question 3

**What recourse does that person actually have?**

*You may Google the Rail Accident Investigation Branch or civil litigation. Think about whether these routes are fast, accessible, and likely to succeed for an ordinary person. If the honest answer is that recourse is limited, say so.*

In [ ]:
available_recourse = """Replace this text with your answer."""

# Minimum 25 words. Name specific routes and assess honestly whether they work in practice.
available_recourse = check_response('Recourse', available_recourse, min_words=25)

In [ ]:
# Display your Section 3 answers before proceeding
display(Markdown('### Section 3 responses accepted'))
display(Markdown(f'**Person at risk**\n\n{person_at_risk}'))
display(Markdown(f'**Consequence of a false negative**\n\n{consequence_of_false_negative}'))
display(Markdown(f'**Available recourse**\n\n{available_recourse}'))
display(Markdown('\n*Proceed to Section 4.*'))

---

## Section 4: The Deployment Recommendation

*About 10 minutes.*

You have the technical evidence. You have worked through who bears the cost. Now make the decision.

>
>#### _Important Quick Reference: Operating threshold_
_**The decision threshold is the predicted probability at which the model switches from "no defect" to "defect".**_
>
>At the default threshold of **0.5**, an image is flagged if the model is 50% or more confident a defect is present.
>
>_You can change this:_
>
>| Threshold | What it means | Effect |
>|:--|:--|:--|
>| 0.3 | Flag if 30% or more confident | More defects caught, more false alarms |
>| 0.5 | Flag if 50% or more confident | Default setting |
>| 0.7 | Flag if 70% or more confident | Fewer false alarms, more missed >defects |
>
>**You explored this in Section 2. Use what you found there.**
>
>If you are keeping the default, say so and explain why 0.5 is the right operating point for this domain.
>
>If you are changing it, state the value and what it achieves.
>

Fill in each variable below. All fields are required and each has a minimum word count. The last field is the most important one.

*(This output is checked as it maps to K26 and B4 in your occupational standard.)*

In [ ]:
# ── Complete each field ──────────────────────────────────────────────────

# Which model do you recommend? (minimum 2 words)
recommended_model = """Replace this text with your answer."""

# What decision threshold will you use, and why? (minimum 10 words)
operating_threshold = """Replace this text with your answer."""

# Why does this model make sense in metric terms? (minimum 20 words)
metric_rationale = """Replace this text with your answer."""

# Why does this model make sense in terms of the people affected? (minimum 20 words)
human_rationale = """Replace this text with your answer."""

# What oversight safeguard must be in place before deployment? (minimum 15 words)
oversight_safeguard = """Replace this text with your answer."""

# If this system produces a false negative and harm results, who is accountable and why? (minimum 15 words)
accountability_statement = """Replace this text with your answer."""

# ── Validate ─────────────────────────────────────────────────────────────
recommended_model      = check_response('Recommended model',      recommended_model,      min_words=2)
operating_threshold    = check_response('Operating threshold',    operating_threshold,    min_words=10)
metric_rationale       = check_response('Metric rationale',       metric_rationale,       min_words=20)
human_rationale        = check_response('Human rationale',        human_rationale,        min_words=20)
oversight_safeguard    = check_response('Oversight safeguard',    oversight_safeguard,    min_words=15)
accountability_statement = check_response('Accountability',       accountability_statement, min_words=15)

print('All fields validated. Generating recommendation...')

In [ ]:
# Build and display the recommendation

recommendation_text = f"""DEPLOYMENT RECOMMENDATION
Unit 7.4 | 7_4_bearing_inspection.ipynb
{'='*60}

RECOMMENDED MODEL
{recommended_model}

OPERATING THRESHOLD
{operating_threshold}

METRIC RATIONALE
{metric_rationale}

HUMAN RATIONALE
{human_rationale}

OVERSIGHT SAFEGUARD REQUIRED BEFORE DEPLOYMENT
{oversight_safeguard}

ACCOUNTABILITY
{accountability_statement}

{'='*60}
Section 3 context
Person at risk           : {person_at_risk[:120]}{'...' if len(person_at_risk)>120 else ''}
Consequence              : {consequence_of_false_negative[:120]}{'...' if len(consequence_of_false_negative)>120 else ''}
Recourse assessment      : {available_recourse[:120]}{'...' if len(available_recourse)>120 else ''}
"""

# Display formatted
display(Markdown('## Your deployment recommendation'))
for line in recommendation_text.split('\n'):
    if line.startswith('===') or line.startswith('---'):
        display(Markdown('---'))
    elif line.isupper() and line.strip():
        display(Markdown(f'**{line}**'))
    else:
        display(Markdown(line))

# Save as plain text
output_path = 'deployment_recommendation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(recommendation_text)

print(f'Recommendation saved to: {output_path}')
print('You can download this file from the JupyterLite file browser (left panel).')

---

## Section 5: Extension

*Optional. About 10 minutes.*
---

Is there a threshold, or a set of conditions, under which you would recommend that **neither model** should be deployed at all?

If yes, describe the conditions precisely. If no, explain why not.

In [ ]:
non_deployment_condition = """Write your answer here if you want to complete the extension.
This cell is optional"""

# This cell is optional. Run it or skip it.
if 'write your answer here' not in non_deployment_condition.lower():
    display(Markdown(f'**Extension response**\n\n{non_deployment_condition}'))
else:
    print('Extension skipped.')

---

*End of notebook. Return to the Rise lesson to complete the knowledge check.*

*Your output is the **deployment_recommendation.txt** file saved in Section 4. You can save it in your learning journal and/or discuss it with your coach and/or line manager.*